# NoisiQ — Week 7 Notebook
**Noise-Aware Quantum Circuit Simulation and Visualization**
*Period: May 4 – May 10, 2026*

---

## Purpose of This Notebook

This notebook serves as the living record and demonstration for Week 7 of the NoisiQ project.

By the end of this notebook you should be able to:
- Apply over/under-rotation coherent errors to a circuit
- Observe systematic error accumulation vs stochastic Pauli noise averaging
- Select a modality preset (superconducting, trapped ion, neutral atom) and auto-configure noise
- Define a custom noise model using the `NoiseModel` protocol
- See the upgraded downstream-impact heatmap halo (replaces error-rate proxy from Week 4)
- Trigger a fan-out propagation graph from the hover menu for any gate
- Plot a fidelity decay curve across circuit depths

---

## Week 7 Milestone Summary

Week 7 adds **coherent control errors**, the `NoiseModel` base class, and **modality presets**.
It also completes two deferred visualization upgrades: the **downstream impact halo** (replacing
the Week 4 error-rate proxy) and the **fan-out propagation graph** (accessible from the hover
tooltip). The **fidelity decay chart** (`plot_fidelity_decay`) is also added to `charts.py`.

---

## Status Tracker

| Task | Owner | Status |
|------|-------|--------|
| `noisiq/noise/noise_model.py` — `NoiseModel` abstract base class / Protocol | TJ | ☐ Todo |
| `noisiq/noise/coherent_errors.py` — `OverRotationError`, `UnderRotationError` | TJ | ☐ Todo |
| `noisiq/noise/modality_presets.py` — `SuperconductingPreset`, `TrappedIonPreset`, `NeutralAtomPreset` | TJ | ☐ Todo |
| `noisiq/backends/many_shot_runner.py` — extend: `downstream_impact_matrix` + `run_depth_sweep()` | TJ | ☐ Todo |
| `noisiq/visualization/heatmap.py` — upgrade halo to use `downstream_impact_matrix` | TJ | ☐ Todo |
| `noisiq/visualization/fanout.py` — fan-out propagation DAG (networkx + matplotlib) | TJ | ☐ Todo |
| `noisiq/visualization/charts.py` — add `plot_fidelity_decay()` | TJ | ☐ Todo |
| `noisiq/visualization/widgets.py` — wire fan-out trigger into hover Output panel | TJ | ☐ Todo |
| `tests/noise/test_coherent_errors.py` | TJ | ☐ Todo |
| All tests passing via `pytest` | TJ | ☐ Todo |
| CI passing on GitHub | TJ | ☐ Todo |
| Week 7 demo section of this notebook complete | TJ | ☐ Todo |

---

## File Build Order

```
1. noisiq/noise/noise_model.py              ← Abstract base / Protocol (no deps)
2. noisiq/noise/coherent_errors.py          ← Over/under-rotation (depends on noise_model + ir)
3. noisiq/noise/modality_presets.py         ← Hardware presets (depends on all noise channel classes)
4. noisiq/backends/many_shot_runner.py      ← Extend: downstream_impact_matrix + run_depth_sweep
5. noisiq/visualization/heatmap.py          ← Upgrade halo to downstream_impact_matrix
6. noisiq/visualization/fanout.py           ← Fan-out DAG (depends on networkx + circuit + AggregateResult)
7. noisiq/visualization/charts.py           ← Add plot_fidelity_decay (depends on List[AggregateResult])
8. noisiq/visualization/widgets.py          ← Wire fan-out trigger into hover panel
9. tests/noise/test_coherent_errors.py      ← Tests for (2)
```

---

## Physics Requirements for Week 7

**Coherent (over-rotation) error:**
- Target gate `G` followed by systematic `R_z(ε)` rotation offset
- Error accumulates linearly with gate count: after N gates, total error angle ≈ N·ε
- Distinguishable from stochastic noise: does not average out over many shots

**Stochastic (Pauli) noise:**
- Error probability per shot is `p`; mean error rate stays constant
- Averages out in expectation values; shows up as decoherence, not systematic shift

**Downstream impact metric (many_shot_runner.py extension):**
- For each error event in a shot: run `PauliFrameTracker` forward from that gate to end of circuit
- Count how many qubits are non-I at the end; store in `downstream_impact_matrix[qubit, op_idx]`
- Skip propagation when sampled Pauli is I (optimization — most shots at low p)
- `run_depth_sweep(circuit, n_shots, noise_config, seed)`: runs `ManyShotRunner` on sub-circuits
  of depth 1..N, returns `List[AggregateResult]` for fidelity decay plotting

**Heatmap halo upgrade:**
- Replace `error_rate_matrix` proxy with `downstream_impact_matrix`
- Normalize: gate with maximum total downstream qubit interactions → brightest red
- Zero downstream interactions → light blue halo

**Fidelity decay:**
- Fidelity proxy = fraction of shots with zero total errors at circuit depth d
- Plot: x-axis = gate depth (1..N), y-axis = estimated fidelity

**Modality preset parameter ranges:**

| Platform | T1 | T2 | Single-qubit gate error | Two-qubit gate error |
|----------|-----|-----|------------------------|---------------------|
| Superconducting | 50–100 µs | 50–100 µs | 0.1–0.3% | 0.5–1% |
| Trapped ion | ~minutes | ~seconds | 0.05–0.1% | 0.3–1% |
| Neutral atom | ~seconds | 0.1–1 s | 0.5–1% | 0.5–1% |

---

## Notes and Decisions Log

| Date | Note | Name |
|------|------|------|
| 2026-05-04 | Week 7 started. Modality presets are a high-value demo feature for interviews. | DS |
| 2026-04-21 | Added downstream_impact_matrix backend extension, heatmap halo upgrade, fanout.py, and fidelity decay chart to this week. These were deferred from Week 4/5 planning. | DS |
| | | |

# Installation Instructions (Developer)

```bash
pip install -e .
```

In [ ]:
import noisiq as nq
import numpy as np
import matplotlib.pyplot as plt

print(f"noisiq {nq.__version__} imported successfully")

# Build a circuit with repeated X gates (accumulates coherent error)
n_reps = 20
circuit = nq.Circuit(n_qubits=1, name="coherent_error_demo")
for _ in range(n_reps):
    circuit.add_gate(nq.ir.X, qubits=[0])

# 1. Coherent-only: systematic over-rotation by epsilon per gate
epsilon = 0.05
coherent_noise = nq.noise.OverRotationError(gate=nq.ir.X, epsilon=epsilon)

# 2. Stochastic-only: depolarizing with equivalent error magnitude
p_equiv = epsilon ** 2 / 2
stochastic_noise = nq.noise.DepolarizingChannel(p=p_equiv)

# Run both simulations
n_shots = 1000
backend = nq.backends.BackendSelector.select(circuit, coherent_noise)
coherent_result = backend.run(circuit, noise_model=coherent_noise, n_shots=n_shots)
stochastic_result = backend.run(circuit, noise_model=stochastic_noise, n_shots=n_shots)
print(f"Coherent error P(|0⟩): {coherent_result.outcome_probabilities}")
print(f"Stochastic error P(|0⟩): {stochastic_result.outcome_probabilities}")

# 3. Modality presets
print("\n--- Modality Presets ---")
for preset_cls in [nq.noise.SuperconductingPreset, nq.noise.TrappedIonPreset, nq.noise.NeutralAtomPreset]:
    preset = preset_cls()
    info = preset.describe()
    print(f"{info['name']}: T1={info['T1']}, T2={info['T2']}, gate_error={info['single_qubit_gate_error']}")

# 4. Downstream impact heatmap (upgraded from error-rate proxy)
print("\n--- Downstream impact heatmap ---")
noise_config = {i: stochastic_noise.to_pauli_error() for i in range(len(circuit.operations))}
runner = nq.backends.ManyShotRunner()
agg_result = runner.run(circuit, n_shots=500, noise_config=noise_config, seed=42)
print(f"downstream_impact_matrix shape: {agg_result.downstream_impact_matrix.shape}")
nq.visualization.plot_error_heatmap(agg_result, circuit)

# 5. Fan-out graph from gate 0 (also accessible via hover menu click)
print("\n--- Fan-out propagation graph ---")
nq.visualization.plot_fanout_graph(circuit, op_idx=0, result=agg_result)

# 6. Fidelity decay across circuit depths
print("\n--- Fidelity decay ---")
depth_results = nq.backends.ManyShotRunner().run_depth_sweep(
    circuit, n_shots=500, noise_config=noise_config, seed=42
)
nq.visualization.charts.plot_fidelity_decay(depth_results)